In [1]:
import os
os.getcwd()
os.chdir('/Users/tommasodifrancesco/Desktop/Lasso_paper/Empirical/scripts/lasso_11_2025')
from stage1 import calculate_r_squared

In [2]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from itertools import product
from tqdm import tqdm
from scipy.optimize import minimize, curve_fit

In [3]:
def alm_return(x_tm1, x_t, kappa, intercept):
    """
    Pure ALM mapping: no data logic, no time indices.
    """
    return (
        np.log(1 - kappa * np.exp(x_tm1))
        - np.log(1 - kappa * np.exp(x_t))
        + intercept + np.random.normal(0, 0.001, size=len(x_t))
    )


In [4]:
kappa_true = 0.3
intercept_true = 0.0
predictions = np.random.normal(0, 0.01, size=2000)
x_tm1 = predictions[:-1]
x_t = predictions[1:]
r_t = alm_return(x_tm1, x_t, kappa_true, intercept_true)
#add a 0 at the and of r_t
r_t = np.hstack([r_t, 0])
#stack the x_tm1 and x_t for curve fitting
X_fit = np.vstack([x_tm1, x_t])
stage2 = {'vwretd': pd.Series(r_t), 'predictions': pd.Series(predictions)}


In [5]:
def compute_alm_returns(predictions, kappa, intercept):
    pred_t, pred_t1 = predictions[:-1], predictions[1:]

    # validity mask
    valid = (1 - kappa * np.exp(pred_t) > 0) & (1 - kappa * np.exp(pred_t1) > 0)
    #valid = (1 - kappa * np.exp(- 1 / kappa * (1 - kappa) * pred_t) > 0) & (1 - kappa * np.exp(- 1 / kappa * (1 - kappa) *  pred_t1) > 0)

    # initialize output array
    alm = np.full(len(pred_t), np.nan)

    # compute only valid entries
    alm[valid] = (
        np.log(1 - kappa * np.exp(pred_t[valid]))
        - np.log(1 - kappa * np.exp(pred_t1[valid]))
        + intercept
    )
    return alm

In [6]:
def estimate_kappa_curve_fit(stage2):
    """
    Estimate kappa and intercept using scipy's curve_fit.
    
    Returns
    -------
    popt : array
        Optimal parameters [kappa, intercept]
    pcov : 2D array
        Covariance matrix of parameters
    """
    r = stage2['vwretd'].values[:-1] # up to r_{t-1}
    pred_t = stage2['predictions'].values[:-1] # up to r^e_{t-1}
    pred_t1 = stage2['predictions'].values[1:] # up to r^e_{t}
    
    def alm_model(x, kappa, intercept):
        pred_t, pred_t1 = x
        return np.log(1 - kappa * np.exp(pred_t)) - np.log(1 - kappa * np.exp(pred_t1)) + intercept
        #return np.log(1 - kappa * np.exp(- 1/kappa *pred_t * (1 -kappa))) - np.log(1 - kappa * np.exp(- 1/kappa *pred_t1 * (1 -kappa))) + intercept
    
    try:
        return curve_fit(
            alm_model, 
            (pred_t, pred_t1), 
            r,
            p0=[0.5, 0.0],
            bounds=([0, -1], [1, 1])
        )
    except Exception as e:
        raise RuntimeError(f"Curve fitting failed: {e}")

In [7]:
estimate_kappa_curve_fit(stage2)

(array([3.00661698e-01, 2.07064833e-05]),
 array([[ 5.73098494e-07, -3.87053728e-12],
        [-3.87053728e-12,  5.02970913e-10]]))

In [8]:
def compute_stage2_r_squared(stage2_input, min_train_size=100):
    """
    Compute in-sample and out-of-sample R² for Stage 2.
    
    In-sample R²:
    - Estimate kappa and intercept on full sample
    - Calculate fitted values on full sample
    - Compute R²
    
    Out-of-sample R²:
    - Use expanding window: for each time t, estimate kappa and intercept on data up to t-1
    - Predict return at time t using this kappa
    - Compute R² on all OOS predictions
    
    Parameters
    ----------
    stage2_input : pd.DataFrame
        Must have columns 'vwretd' (actual returns) and 'predictions' (Stage 1 predictions)
    min_train_size : int
        Minimum number of observations needed to estimate kappa
        
    Returns
    -------
    dict
        Contains r2_insample, r2_oos, kappa_full, intercept_full, and their t-stats
    """
    
    # ===== IN-SAMPLE R² =====
    # Estimate kappa on FULL sample
    try:
        popt_full, pcov_full = estimate_kappa_curve_fit(stage2_input)
        kappa_full, intercept_full = popt_full
        se_full = np.sqrt(np.diag(pcov_full))
        
        # Handle zero or near-zero standard errors
        if se_full[0] < 1e-30:
            kappa_tstat = np.nan  # Can't compute t-stat
        else:
            kappa_tstat = kappa_full / se_full[0]
        
        if se_full[1] < 1e-30:
            intercept_tstat = np.nan
        else:
            intercept_tstat = intercept_full / se_full[1]
        
        # Generate fitted values on FULL sample
        preds_full = stage2_input['predictions'].values
        alm_fitted = compute_alm_returns(preds_full, kappa_full, intercept_full)
        
        # Calculate in-sample R²
        y_full = stage2_input['vwretd'].values[:-1]
        
        # Check if we have valid fitted values
        if len(alm_fitted) == 0:
            return {
                'r2_insample': np.nan,
                'r2_oos': np.nan,
                'kappa': kappa_full,
                'kappa_tstat': kappa_tstat,
                'intercept': intercept_full,
                'intercept_tstat': intercept_tstat,
                'error': "No valid ALM fitted values (constraint violations)"
            }
        
        r2_insample = calculate_r_squared(y_full, alm_fitted)
        
    except Exception as e:
        return {
            'r2_insample': np.nan,
            'r2_oos': np.nan,
            'kappa': np.nan,
            'kappa_tstat': np.nan,
            'intercept': np.nan,
            'intercept_tstat': np.nan,
            'error': f"Full sample estimation failed: {e}"
        }
    
    # ===== OUT-OF-SAMPLE R² =====
    # Use train-test split
    oos_predictions = []
    oos_actuals = []
    test_size = 0.2
    n = len(stage2['vwretd'])
    n_test = int(n * test_size)

    X_train = stage2['predictions'].values[:-n_test]
    y_train = stage2['vwretd'].values[:-n_test]
    train_data = pd.DataFrame({'predictions': X_train, 'vwretd': y_train})
    X_test = stage2['predictions'].values[-n_test:]
    y_test = stage2['vwretd'].values[-n_test:]
    test_data = pd.DataFrame({'predictions': X_test, 'vwretd': y_test})


    #get parameters on train data
    try:
        popt_oos, _ = estimate_kappa_curve_fit(train_data)
        kappa_oos, intercept_oos = popt_oos
    except Exception as e:
        print(f"OOS parameter estimation failed: {e}")
        kappa_oos, intercept_oos = np.nan, np.nan

    #generate OOS predictions
    if not np.isnan(kappa_oos):
        preds_oos = test_data['predictions'].values
        alm_oos = compute_alm_returns(preds_oos, kappa_oos, intercept_oos)
        
        # Store valid OOS predictions and actuals
        oos_predictions.extend(alm_oos)
        oos_actuals.extend(test_data['vwretd'].values[:-1])  # Align lengths
    
    # Calculate OOS R²
    if len(oos_predictions) > 0:
        oos_predictions = np.array(oos_predictions)
        oos_actuals = np.array(oos_actuals)
        r2_oos = calculate_r_squared(oos_actuals, oos_predictions)
    else:
        r2_oos = np.nan
    
    return {
        'r2_insample': r2_insample,
        'r2_oos': r2_oos,
        'kappa': kappa_full,
        'kappa_tstat': kappa_tstat,
        'intercept': intercept_full,
        'intercept_tstat': intercept_tstat,
    }

In [9]:
res = compute_stage2_r_squared(stage2, min_train_size=100)
res


{'r2_insample': np.float64(0.9747654589823684),
 'r2_oos': np.float64(0.9784241243651548),
 'kappa': np.float64(0.3006616979371087),
 'kappa_tstat': np.float64(397.15814361266916),
 'intercept': np.float64(2.070648327850913e-05),
 'intercept_tstat': np.float64(0.9232831522100245)}